In [1]:
import snappy
import networkx as nx
from knot_graphs import *

import pdb

# We want to identify pinch moves (planar, non-orientable band moves)

In [ ]:
draw_adjacency_graph(snappy.Link([[8,3,1,4],[2,6,3,5],[6,2,7,1],[4,7,5,8]]))

> /home/julesg2/KnotGeography/knot_graphs.py(55)adjacency_graph()
     53     #pdb.set_trace()
     54     for u,v in set(G.edges()):
---> 55         pdb.set_trace()
     56         num_uv_edges = G.number_of_edges(u,v) + G.number_of_edges(v,u)
     57         if num_uv_edges == 1:



ipdb>  print(u)


1


ipdb>  print(v)


2


...

{0: {'head_strand': 0, 'tail_strand': 1}}


ipdb>  G.get_edge_data(1,2)


{0: {'head_strand': 0, 'tail_strand': 1}}


ipdb>  G.get_edge_data(2,1)


{0: {'head_strand': 0, 'tail_strand': 1}}


In [ ]:
K = snappy.Link("T(2,3)")

In [ ]:
#K.view()
#%gui tk

In [ ]:
draw_adjacency_graph(K, rad_increment=0.15)

In [ ]:
G = adjacency_graph(K)

In [ ]:
G.nodes()

In [ ]:
G.edges()

In [ ]:
G.get_edge_data(0, 1)

In [ ]:
K.crossings

In [ ]:
[c.label for c in K.crossings]

In [ ]:
[c.adjacent for c in K.crossings]

In [ ]:
def planar_band_targets(edge, planar_emb):
    face_nodes = planar_emb.traverse_face(edge[0], edge[1])
    return [(face_nodes[i], face_nodes[(i+1) % len(face_nodes)]) for i in range(len(face_nodes))]

In [ ]:
def list_planar_arc_pairs(knot):
    graph = adjacency_graph(knot)
    is_planar, planar_emb = nx.check_planarity(graph)
    if not is_planar:
        return "Knot graph isn't planar!"

    planar_band_arcs = {}
    for e in planar_emb.edges():
        planar_band_arcs[e] = planar_band_targets(e, planar_emb)[1:] # all other edges in the right face associated to e
    return planar_band_arcs

In [ ]:
list_planar_arc_pairs(K)

### TODO - there should be nine total planar arc-pairs
(18 entries in above dict since each band is identified by both of its ends)
<div>
<center>
    <img src="images/pinch_sketch.jpg" width="800"/>
</center>
</div>

What about cases like the arc 2 to arc 5 band?

Its orientable band is twisted, so is the negative twist also orientable?

Are both the negative twist and the 0 twist non-orientable?

### TODO - identify which twists are non-orientable (hence a pinch move) for each planar band (arc pair) above

Write a function that takes a pair of arcs and returns which twist numbers give non-orientable bands.

### Idea: orientable bands yield links, non-orientable bands yield knots.
- Pick any adjacent pair of strands
- Try -1, 0, 1-twist bands on them
    - Take complement + identify resulting manifold using `snappy.Manifold.identify`
    - If complement is a link then the band is orientable.
    - If complement is a knot then the band is non-orientable.

In [ ]:
def find_pinch_move(K):
    graph = adjacency_graph(K)
    planar_arc_pairs = list_planar_arc_pairs(K)

    #take first pair of adjacent arcs
    arc0, adj_arcs = list(planar_arc_pairs.items())[0]
    arc1 = adj_arcs[0]
    pdb.set_trace()
    for twist in [-1,0,1]:
        # get (crossing, strand)'s from edges' (crossing, crossing) in arc1/arc2
        c0 = arc0[0]
        c1 = arc1[0]

        ed0 = graph.get_edge_data(arc0[0], arc0[1])
        if ed0 is None:
            print(f'arc0 {arc0} not found in graph')
            continue
        edge0 = list(ed0.values())[0]
        s0 = edge0['tail_strand']

        ed1 = graph.get_edge_data(arc1[0], arc1[1])
        if ed1 is None:
            print(f'arc1 {arc1} not found in graph')
            continue
        edge1 = list(ed1.values())[0]
        s1 = edge1['tail_strand']
        try:
            S = Cobordism(K)
            
            # do the band move
            S.band_move(twist, (c0, s0), (c1, s1))

            # identify
            K_prime = S.links[-1]
            snappy_K_prime = snappy.Link(K_prime.PD_code())
            
            #writhe_prime = snappy_K_prime.writhe()
            #normal_euler = writhe_prime - writhe
            #es.append(normal_euler)

            complement = snappy_K_prime.exterior()
            result = complement.identify(extends_to_link=True)

            if len(result) == 0:
                print(f'band ({twist}, ({c0},{s0}), ({c1},{s1})) could not be identified')
            else:
                print(result)
        except:
            print(f'band ({twist}, ({c0},{s0}), ({c1},{s1})) threw an error')
            continue

In [ ]:
find_pinch_move(K)